## Import Files in Colab

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sequences.fasta to sequences.fasta
Saving test.tsv to test.tsv
Saving train.tsv to train.tsv


In [ ]:
import pandas as pd

train_data = pd.read_csv('train.tsv', sep='\t')
test_data = pd.read_csv('test.tsv', sep='\t')

print("Train data:", train_data.head())
print("Test data:", test_data.head())

Train data:             id secondary_structure
0   3KVH_LYS_6                   .
1   3KVH_GLN_7                   E
2   3KVH_ILE_8                   E
3   3KVH_SER_9                   .
4  3KVH_ARG_10                   H
Test data:             id
0   3JRN_LYS_8
1   3JRN_TYR_9
2  3JRN_ASP_10
3  3JRN_VAL_11
4  3JRN_PHE_12


## Loading ESM model

In [ ]:
!pip install transformers biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 22.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModel, EsmForTokenClassification, TrainingArguments, Trainer
import torch

In [ ]:
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Read Sequences from fasta file & split training ids



In [ ]:
from Bio import SeqIO

fasta_file = "sequences.fasta"

def read_fasta(fasta_path):
    id_to_seq = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        id_to_seq[record.id] = str(record.seq)
    print(f"{len(id_to_seq)} protein sequences")
    return id_to_seq

id_to_seq = read_fasta(fasta_file)


split_cols = train_data['id'].str.split('_', expand=True) # split id in training data into columns to match protein ids in training & fasta files
train_data['protein_id'] = split_cols[0]
train_data['amino_acid'] = split_cols[1]
train_data['pos'] = split_cols[2].astype(int)

train_data.head()


8693 protein sequences


,id,secondary_structure,protein_id,amino_acid,pos
0,3KVH_LYS_6,.,3KVH,LYS,6
1,3KVH_GLN_7,E,3KVH,GLN,7
2,3KVH_ILE_8,E,3KVH,ILE,8
3,3KVH_SER_9,.,3KVH,SER,9
4,3KVH_ARG_10,H,3KVH,ARG,10


## Tokenize sequences, input into ESM model and get embeddings

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm

cls_embeddings = []
structure_labels = []


for protein_id, group in tqdm(train_data.groupby('protein_id')): # accounts for mismatch in fasta sequences and training data sequences, groups amino acids of same proteins
    if protein_id not in id_to_seq:
        continue

    sequence = id_to_seq[protein_id] # takes sequence from sequences fasta dictionary based on protein_id from train data

    # see esm demo on discussion slides
    inputs = tokenizer(sequence, return_tensors="pt", truncation=True, add_special_tokens=True) # tokenizes protein sequences from fasta file for input in ESM

    with torch.no_grad():
        outputs = model(**inputs) # inputs tokenized sequences into ESM model

    token_embeddings = outputs.last_hidden_state.squeeze(0) # Shape: (1 (batch size), sequence_length + 2, hidden_size), gets embeddings from esm model

    for _, row in group.iterrows(): # go through all amino acids that correspond to same protein sequence
        aa_pos = int(row['pos'])
        if aa_pos + 1 >= token_embeddings.shape[0]:
            continue  # skip invalid positions
        embedding = token_embeddings[aa_pos + 1] # embedding from ESM model for the amino acid
        cls_embeddings.append(embedding.numpy()) # sequences -> tokenized -> token embeddings -> embedding -> classifier embeddings to be used as input for ML model
        structure_labels.append(row['secondary_structure']) # secondary_structure label given in training data -> use as input for ML model

#print("Embeddings, features: ", x.shape) # try cutting down on number of embeddings to decrease RAM usage

100%|██████████| 6159/6159 [17:25<00:00,  5.89it/s]


NameError: name 'x' is not defined

In [ ]:
# only run when saving embeddings for the first time
x = np.array(cls_embeddings)
y = np.array(structure_labels)

import joblib
joblib.dump((x,y),"embeddings.pkl")

NameError: name 'np' is not defined

## Start from here after getting embeddings

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import joblib

embeddings_file = "/content/drive/MyDrive/embeddings.pkl"
x, y = joblib.load(embeddings_file)

Mounted at /content/drive


In [ ]:
print("Number of embeddings:", x.shape[0])

Number of embeddings: 2700223


## Train ML model

In [ ]:
from sklearn.utils import resample

x_small, y_small = resample(x, y, n_samples=500000, random_state=42) # use subset of embeddings (so colab doesn't crash)
x_train, x_val, y_train, y_val = train_test_split(x_small, y_small, test_size=0.2, random_state=42) # use 20% of train data as test, 80% for training

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(x_train, y_train) # train random forest model

y_pred = clf.predict(x_val)
print("Accuracy: ",accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Accuracy:  0.70254

Classification Report:
               precision    recall  f1-score   support

           .       0.55      0.59      0.57     17015
           B       0.96      0.31      0.47      1236
           E       0.67      0.86      0.76     23082
           G       0.95      0.29      0.44      3960
           H       0.76      0.92      0.83     32750
           I       0.95      0.30      0.45       543
           P       0.89      0.28      0.42      1850
           S       0.90      0.32      0.47      8320
           T       0.74      0.47      0.57     11244

    accuracy                           0.70    100000
   macro avg       0.82      0.48      0.55    100000
weighted avg       0.73      0.70      0.68    100000



## Embeddings for test data

In [ ]:
split_cols = test_data['id'].str.split('_', expand=True)
test_data['protein_id'] = split_cols[0]
test_data['amino_acid'] = split_cols[1]
test_data['pos'] = split_cols[2].astype(int)
test_data.head()

,id,protein_id,amino_acid,pos
0,3JRN_LYS_8,3JRN,LYS,8
1,3JRN_TYR_9,3JRN,TYR,9
2,3JRN_ASP_10,3JRN,ASP,10
3,3JRN_VAL_11,3JRN,VAL,11
4,3JRN_PHE_12,3JRN,PHE,12


In [ ]:
test_embeddings = []
test_ids = []


for protein_id, group in test_data.groupby('protein_id'):
    if protein_id not in id_to_seq:
        continue

    sequence = id_to_seq[protein_id]

    inputs = tokenizer(sequence, return_tensors="pt", truncation=True, add_special_tokens=True)

    with torch.no_grad():
        outputs = model(**inputs)

    token_embeddings = outputs.last_hidden_state.squeeze(0)

    for _, row in group.iterrows():
        aa_pos = int(row['pos'])
        if aa_pos + 1 >= token_embeddings.shape[0]:
            continue  # skip invalid positions
        embedding = token_embeddings[aa_pos + 1]
        test_embeddings.append(embedding.numpy())
        test_ids.append(row['id'])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
X_test = np.array(test_embeddings)
print("Number of embeddings:", x.shape[0])

Number of embeddings: 2700223


In [ ]:
# only run first time saving test embeddings
test_ids = np.array(test_ids)

joblib.dump((X_test, test_ids), "test_embeddings.pkl")

['test_embeddings.pkl']

## Read in test embeddings & save predictions file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import joblib

test_embeddings_file = "/content/drive/MyDrive/test_embeddings.pkl"
x, y = joblib.load(test_embeddings_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X_test = x
test_ids = y
y_test_pred = clf.predict(X_test) # predictions using test embeddings

In [ ]:
submission_df = pd.DataFrame({
    "id": test_ids,
    "secondary_structure": y_test_pred
})

submission_df.to_csv("predictions.csv", sep="\t", index=False, header=True)